# LeetCode 1952 — Three Divisors bằng Python

**Link đề:** https://leetcode.com/problems/three-divisors/
**Độ khó:** Easy · **Chủ đề:** Math, Enumeration, Number Theory

```python
class Solution:
    def isThree(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **ước số** (divisor) và cách kiểm tra bằng phép chia lấy dư `n % d == 0`.
2. Đi từ đếm ước `O(n)` → đếm theo **cặp ước** tới `√n` (`O(√n)`).
3. Rút ra insight toán học: *có đúng 3 ước ⇔ `n = p²` với `p` nguyên tố*.
4. Biết `math.isqrt`, kiểm tra số chính phương, kiểm tra số nguyên tố, **sàng Eratosthenes**.
5. Trace, chứng minh, Time/Space, lỗi hay gặp (đặc biệt `n = 1`) và các biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — có đúng ba ước dương không?

Cho số nguyên `n`. Trả `True` nếu `n` có **đúng 3 ước dương**, ngược lại `False`.

Số nguyên `m` là **ước** của `n` nếu tồn tại số nguyên `k` với `n = k · m`.

| `n` | Các ước | Output |
|---:|---|:---:|
| `2` | 1, 2 | `False` (chỉ 2 ước) |
| `4` | 1, 2, 4 | `True` |

**Constraints:** `1 <= n <= 10^4`.

Gợi ý từ đề: *cẩn thận với `n = 1`* — một số lời giải sai ở đây.

In [1]:
for n in [1, 2, 4, 6, 9]:
    divisors = [d for d in range(1, n + 1) if n % d == 0]
    print(n, divisors, "->", len(divisors), "ước")

1 [1] -> 1 ước
2 [1, 2] -> 2 ước
4 [1, 2, 4] -> 3 ước
6 [1, 2, 3, 6] -> 4 ước
9 [1, 3, 9] -> 3 ước


## 2. Nền tảng Python và toán cần dùng

### 2.1. `%` — kiểm tra chia hết

`n % d == 0` nghĩa là `d` chia hết `n` (`d` là ước của `n`). `n // d` là thương — cũng là một ước, "bạn cặp" của `d`.

In [2]:
n = 12
for d in [1, 2, 3, 5, 12]:
    if n % d == 0:
        print(f"{n} % {d} = 0 -> {d} là ước, 'bạn cặp' là {n // d}")
    else:
        print(f"{n} % {d} = {n % d} -> {d} không phải ước")

12 % 1 = 0 -> 1 là ước, 'bạn cặp' là 12
12 % 2 = 0 -> 2 là ước, 'bạn cặp' là 6
12 % 3 = 0 -> 3 là ước, 'bạn cặp' là 4
12 % 5 = 2 -> 5 không phải ước
12 % 12 = 0 -> 12 là ước, 'bạn cặp' là 1


### 2.2. Cặp ước và vì sao chỉ cần tới `√n`

Nếu `d` là ước thì `n // d` cũng là ước, và trong hai số đó **có một số ≤ √n**. Ví dụ `n = 36`:

| d | 1 | 2 | 3 | 4 | **6** |
|---|---|---|---|---|---|
| n // d | 36 | 18 | 12 | 9 | **6** |

Duyệt `d` từ 1 tới 6 (= √36) là thấy **mọi** ước. Khi `d == n // d` (ở đây `6`), cặp chỉ là **một** ước — đây là chỗ số chính phương có số ước **lẻ**.

In [3]:
n = 36
d = 1
while d * d <= n:           # d <= sqrt(n) viết bằng số nguyên: d*d <= n
    if n % d == 0:
        print(d, "x", n // d, "(trùng)" if d == n // d else "")
    d += 1

1 x 36 
2 x 18 
3 x 12 
4 x 9 
6 x 6 (trùng)


### 2.3. `math.isqrt` — căn bậc hai nguyên

`math.isqrt(n)` trả số nguyên `r` lớn nhất với `r*r <= n`. Dùng nó thay `int(n ** 0.5)` để tránh sai số số thực. `n` là **số chính phương** ⇔ `isqrt(n) ** 2 == n`.

In [4]:
import math

for n in [4, 8, 9, 10, 10000]:
    r = math.isqrt(n)
    print(n, "isqrt =", r, "chính phương?", r * r == n)
print(10 ** 0.5, math.isqrt(10))   # 3.162... và 3

4 isqrt = 2 chính phương? True
8 isqrt = 2 chính phương? False
9 isqrt = 3 chính phương? True
10 isqrt = 3 chính phương? False
10000 isqrt = 100 chính phương? True
3.1622776601683795 3


### 2.4. Số nguyên tố

`p` là **số nguyên tố** (prime) nếu `p ≥ 2` và chỉ có 2 ước: `1` và `p`. Kiểm tra bằng cách thử chia cho mọi `d` từ 2 tới `√p`.

Lưu ý: `1` **không** phải số nguyên tố (chỉ có 1 ước).

In [5]:
def is_prime(p):
    if p < 2:
        return False
    d = 2
    while d * d <= p:
        if p % d == 0:
            return False
        d += 1
    return True

print([p for p in range(1, 30) if is_prime(p)])

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


## 3. Tính tay — những số nào có đúng 3 ước?

Liệt kê số ước của `n = 1..30`:

In [6]:
def count_divisors_naive(n):
    return sum(1 for d in range(1, n + 1) if n % d == 0)

table = {n: count_divisors_naive(n) for n in range(1, 31)}
print(table)
print("Có đúng 3 ước:", [n for n, c in table.items() if c == 3])

{1: 1, 2: 2, 3: 2, 4: 3, 5: 2, 6: 4, 7: 2, 8: 4, 9: 3, 10: 4, 11: 2, 12: 6, 13: 2, 14: 4, 15: 4, 16: 5, 17: 2, 18: 6, 19: 2, 20: 6, 21: 4, 22: 4, 23: 2, 24: 8, 25: 3, 26: 4, 27: 4, 28: 6, 29: 2, 30: 8}
Có đúng 3 ước: [4, 9, 25]


Chỉ có `4, 9, 25` — đều là **bình phương của số nguyên tố** (`2², 3², 5²`). Còn `16 = 4²` có 5 ước (1, 2, 4, 8, 16) vì 4 không nguyên tố. Giả thuyết này sẽ được chứng minh ở mục 5.

## 4. Cách 1 — Baseline: đếm mọi ước từ 1 tới n

In [7]:
def is_three_naive(n):
    count = 0
    for d in range(1, n + 1):
        if n % d == 0:
            count += 1
    return count == 3

print(is_three_naive(2), is_three_naive(4), is_three_naive(1), is_three_naive(16))   # False True False False

False True False False


Chi phí: `n` phép chia → `O(n)`. Với `n ≤ 10^4` vẫn qua, nhưng nếu `n` tới `10^12` thì hết giờ. Cải tiến dễ: **dừng sớm khi `count > 3`**, nhưng worst case (số nguyên tố) vẫn `O(n)`.

### 4.1. Cách 2 — đếm theo cặp ước tới `√n`

Mỗi `d ≤ √n` chia hết `n` đóng góp **2** ước (`d` và `n // d`), trừ khi `d == n // d` thì đóng góp **1**.

In [8]:
def count_divisors_sqrt(n):
    count = 0
    d = 1
    while d * d <= n:
        if n % d == 0:
            count += 1 if d * d == n else 2
        d += 1
    return count

def is_three_sqrt(n):
    return count_divisors_sqrt(n) == 3

for n in [1, 2, 4, 9, 16, 36, 49]:
    print(n, count_divisors_sqrt(n), is_three_sqrt(n))

1 1 False
2 2 False
4 3 True
9 3 True
16 5 False
36 9 False
49 3 True


Cách 2 chỉ tốn `√n` vòng (`≤ 100` với `n ≤ 10^4`). Nhưng ta vẫn **đếm** — trong khi chỉ cần biết số ước có bằng 3 không. Có cách nhìn cấu trúc thay vì đếm không?

## 5. Bước đột phá — đúng 3 ước ⇔ `n = p²`, `p` nguyên tố

**Lập luận:**
1. Ước đi theo cặp `(d, n/d)`. Số ước là **lẻ** ⇔ có một cặp trùng `d = n/d` ⇔ `n` là **số chính phương** `n = r²`.
2. 3 là số lẻ → `n = r²`. Các ước chắc chắn có: `1`, `r`, `r²` (ba số khác nhau khi `r ≥ 2`).
3. Đúng 3 ước ⇔ **không** có ước nào khác. Nếu `r` có ước `q` với `1 < q < r` thì `q` cũng là ước của `n` → ít nhất 4 ước. Vậy `r` không có ước nào ở giữa ⇔ `r` **nguyên tố**.
4. Trường hợp `r = 1` (`n = 1`): chỉ có 1 ước → `False` (khớp vì 1 không nguyên tố).

Kết luận: `isThree(n) ⇔ n là số chính phương và √n là số nguyên tố`.

In [9]:
# Kiểm chứng giả thuyết trên toàn miền đề bài (1..10^4) bằng baseline cách 2
for n in range(1, 10001):
    r = math.isqrt(n)
    by_formula = (r * r == n) and is_prime(r)
    assert by_formula == is_three_sqrt(n), n
print("Công thức p² khớp với đếm ước trên mọi n từ 1 tới 10000")

Công thức p² khớp với đếm ước trên mọi n từ 1 tới 10000


## 6. Lời giải chính — giảng từng đoạn

**Đoạn 1 — số chính phương?** `r = math.isqrt(n)`; nếu `r * r != n` → `False`.

**Đoạn 2 — `r` nguyên tố?** `r < 2` → `False` (chặn `n = 1`). Thử chia `r` cho `d = 2, 3, …` tới `d*d <= r`.

**Đoạn 3 — trả kết quả.**

In [10]:
def is_three(n):
    r = math.isqrt(n)                 # Đoạn 1
    if r * r != n:
        return False
    if r < 2:                         # Đoạn 2: chặn n = 1
        return False
    d = 2
    while d * d <= r:
        if r % d == 0:
            return False              # r có ước ở giữa -> n có >= 4 ước
        d += 1
    return True                       # Đoạn 3

print([n for n in range(1, 200) if is_three(n)])   # [4, 9, 25, 49, 121, 169]

[4, 9, 25, 49, 121, 169]


## 7. TRACE từng bước

| n | r = isqrt(n) | r² == n? | r < 2? | thử chia r | Kết quả |
|---:|:---:|:---:|:---:|---|:---:|
| 2 | 1 | 1 ≠ 2 | | | False |
| 4 | 2 | có | không | không có d (2·2 > 2) | True |
| 1 | 1 | có | **có** | | False |
| 16 | 4 | có | không | d=2: 4 % 2 = 0 | False |
| 49 | 7 | có | không | d=2: dư 1 → d=3: 9 > 7 dừng | True |
| 50 | 7 | 49 ≠ 50 | | | False |

In [11]:
def trace_is_three(n):
    r = math.isqrt(n)
    print(f"n = {n}: r = isqrt(n) = {r}, r*r = {r * r}")
    if r * r != n:
        print("  không phải số chính phương -> False")
        return False
    if r < 2:
        print("  r < 2 (n = 1) -> chỉ có 1 ước -> False")
        return False
    d = 2
    while d * d <= r:
        print(f"  thử d = {d}: {r} % {d} = {r % d}")
        if r % d == 0:
            print(f"  {r} có ước {d} -> n có thêm ước {d} và {n // d} -> False")
            return False
        d += 1
    print(f"  {r} là số nguyên tố -> ước của n: 1, {r}, {n} -> True")
    return True

for n in [2, 4, 1, 16, 49, 50]:
    trace_is_three(n)

n = 2: r = isqrt(n) = 1, r*r = 1
  không phải số chính phương -> False
n = 4: r = isqrt(n) = 2, r*r = 4
  2 là số nguyên tố -> ước của n: 1, 2, 4 -> True
n = 1: r = isqrt(n) = 1, r*r = 1
  r < 2 (n = 1) -> chỉ có 1 ước -> False
n = 16: r = isqrt(n) = 4, r*r = 16
  thử d = 2: 4 % 2 = 0
  4 có ước 2 -> n có thêm ước 2 và 8 -> False
n = 49: r = isqrt(n) = 7, r*r = 49
  thử d = 2: 7 % 2 = 1
  7 là số nguyên tố -> ước của n: 1, 7, 49 -> True
n = 50: r = isqrt(n) = 7, r*r = 49
  không phải số chính phương -> False


### 7.1. Trace cách đếm cặp ước để so sánh

Với `n = 16`, cách 2 tìm cặp `(1,16)`, `(2,8)`, `(4,4)` → `2 + 2 + 1 = 5` ước. Với `n = 49`: `(1,49)`, `(7,7)` → `2 + 1 = 3`.

In [12]:
def trace_count_pairs(n):
    count, d = 0, 1
    while d * d <= n:
        if n % d == 0:
            add = 1 if d * d == n else 2
            count += add
            print(f"  n={n}: cặp ({d}, {n // d}) -> +{add}, count = {count}")
        d += 1
    return count

print("16 có", trace_count_pairs(16), "ước")
print("49 có", trace_count_pairs(49), "ước")

  n=16: cặp (1, 16) -> +2, count = 2
  n=16: cặp (2, 8) -> +2, count = 4
  n=16: cặp (4, 4) -> +1, count = 5
16 có 5 ước
  n=49: cặp (1, 49) -> +2, count = 2
  n=49: cặp (7, 7) -> +1, count = 3
49 có 3 ước


## 8. Cách 4 — Tiền xử lý bằng sàng Eratosthenes

Vì `n ≤ 10^4` nên `p = √n ≤ 100`. Ta có thể tính trước **mọi** số nguyên tố ≤ 100 bằng **sàng Eratosthenes**, rồi tạo tập `{p²}`. Mỗi truy vấn chỉ còn là `n in tập` — `O(1)`.

**Sàng Eratosthenes:** tạo list `is_p = [True] * (N+1)`, gán `0, 1` là `False`. Với mỗi `i` từ 2 mà `is_p[i]` còn `True`, gạch bỏ các bội `i*i, i*i + i, …`. Những số còn `True` là nguyên tố.

In [13]:
def sieve(N):
    is_p = [True] * (N + 1)
    is_p[0] = is_p[1] = False
    i = 2
    while i * i <= N:
        if is_p[i]:
            for j in range(i * i, N + 1, i):   # range(start, stop, step): bước nhảy i
                is_p[j] = False
        i += 1
    return [x for x in range(N + 1) if is_p[x]]

primes = sieve(100)
print(len(primes), primes[:10], "...", primes[-3:])
SQUARES_OF_PRIMES = {p * p for p in primes}
print(sorted(SQUARES_OF_PRIMES)[:6], "... có", len(SQUARES_OF_PRIMES), "số")

25 [2, 3, 5, 7, 11, 13, 17, 19, 23, 29] ... [83, 89, 97]
[4, 9, 25, 49, 121, 169] ... có 25 số


`range(i * i, N + 1, i)` sinh `i², i² + i, i² + 2i, …` — tham số thứ ba là **bước nhảy** (step). Bắt đầu từ `i²` vì các bội nhỏ hơn (`2i, 3i, …`) đã bị gạch bởi các số nguyên tố nhỏ hơn.

Dùng **set comprehension** `{p * p for p in primes}` để tạo tập (giống list comprehension nhưng dùng ngoặc nhọn).

In [14]:
def is_three_precomputed(n):
    return n in SQUARES_OF_PRIMES

print([n for n in [1, 2, 4, 9, 16, 25, 9409, 10000] if is_three_precomputed(n)])   # 9409 = 97²

[4, 9, 25, 9409]


## 9. Code nộp LeetCode (Python 3)

In [15]:
import math

class Solution:
    def isThree(self, n: int) -> bool:
        r = math.isqrt(n)
        if r * r != n or r < 2:
            return False
        d = 2
        while d * d <= r:
            if r % d == 0:
                return False
            d += 1
        return True


sol = Solution()
assert sol.isThree(2) is False
assert sol.isThree(4) is True
print("2 ví dụ của đề PASS")

2 ví dụ của đề PASS


Bản đếm cặp ước (dễ giải thích trong phỏng vấn nếu chưa nghĩ ra insight):

```python
class Solution:
    def isThree(self, n: int) -> bool:
        count, d = 0, 1
        while d * d <= n:
            if n % d == 0:
                count += 1 if d * d == n else 2
                if count > 3:
                    return False
            d += 1
        return count == 3
```

## 10. Vì sao thuật toán đúng?

**Chiều (⇐):** nếu `n = p²`, `p` nguyên tố, thì một ước `d` của `n` có dạng `p^a` với `a ∈ {0, 1, 2}` (định lý phân tích thừa số nguyên tố duy nhất) → đúng 3 ước `1, p, p²`.

**Chiều (⇒):** nếu `n` có đúng 3 ước:
- Ghép ước thành cặp `(d, n/d)`; các cặp phân biệt đóng góp 2, cặp trùng đóng góp 1. Số ước lẻ ⇒ có cặp trùng ⇒ `n = r²`.
- `1 < r < n` (vì `n ≥ 4`, do `n = 1` chỉ có 1 ước) nên `1, r, n` là ba ước phân biệt — đó là **tất cả**.
- Nếu `r` có ước `q` với `1 < q < r` thì `q` là ước thứ tư của `n` — mâu thuẫn. Vậy `r` nguyên tố.

Thuật toán kiểm tra đúng hai điều kiện "chính phương" và "căn nguyên tố", và `r < 2` loại `n = 1`. Vòng kiểm tra nguyên tố đúng vì: nếu `r` hợp số thì nó có một ước `≤ √r`.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đếm ước 1..n | `O(n)` | `O(1)` | baseline |
| 2. Đếm cặp ước tới √n | `O(√n)` | `O(1)` | cải tiến chuẩn |
| 3. Chính phương + nguyên tố (chính) | `O(n^{1/4})` (kiểm tra nguyên tố `r = √n` tốn `√r`) | `O(1)` | nên nộp |
| 4. Sàng trước tới 100 | tiền xử lý `O(M log log M)` với `M = 100`; mỗi truy vấn `O(1)` | `O(M)` | hợp khi có rất nhiều truy vấn |

`math.isqrt` chạy rất nhanh (coi như `O(1)` với số nhỏ như đề này).

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1: n=2", sol.isThree(2), False)
check("đề 2: n=4", sol.isThree(4), True)
check("n=1 (bẫy)", sol.isThree(1), False)
check("n=3 nguyên tố", sol.isThree(3), False)
check("n=9 = 3²", sol.isThree(9), True)
check("n=16 = 4² (4 hợp số)", sol.isThree(16), False)
check("n=8 = 2³", sol.isThree(8), False)
check("n=12", sol.isThree(12), False)
check("n=9409 = 97² (lớn nhất ≤ 10^4)", sol.isThree(9409), True)
check("n=10000 = 100²", sol.isThree(10000), False)

PASS đề 1: n=2: False
PASS đề 2: n=4: True
PASS n=1 (bẫy): False
PASS n=3 nguyên tố: False
PASS n=9 = 3²: True
PASS n=16 = 4² (4 hợp số): False
PASS n=8 = 2³: False
PASS n=12: False
PASS n=9409 = 97² (lớn nhất ≤ 10^4): True
PASS n=10000 = 100²: False


In [17]:
# Đối chiếu mọi cách trên toàn miền đề bài
funcs = [is_three_naive, is_three_sqrt, is_three, is_three_precomputed, Solution().isThree]
for n in range(1, 10001):
    results = {f(n) for f in funcs}
    assert len(results) == 1, (n, results)
print("Mọi cách khớp nhau trên n = 1..10000; số lượng n thoả:", sum(is_three(n) for n in range(1, 10001)))

Mọi cách khớp nhau trên n = 1..10000; số lượng n thoả: 25


Có đúng **25** giá trị `n ≤ 10^4` thoả — bằng số số nguyên tố ≤ 100. Hợp lý!

## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Chỉ kiểm tra "số chính phương"

In [18]:
def buggy_only_square(n):
    r = math.isqrt(n)
    return r * r == n and n > 1

for n in [4, 16, 36, 1]:
    print(n, "buggy:", buggy_only_square(n), "| đúng:", is_three(n))

4 buggy: True | đúng: True
16 buggy: True | đúng: False
36 buggy: True | đúng: False
1 buggy: False | đúng: False


`16 = 4²` có 5 ước; `36` có 9 ước. Phải kiểm tra thêm **căn là số nguyên tố**.

### Lỗi 2 — Quên `n = 1`

Kiểm tra nguyên tố kiểu "không có ước nào từ 2 tới √r" sẽ coi `r = 1` là nguyên tố (vòng lặp không chạy lần nào) → `isThree(1)` trả `True` sai.

In [19]:
def buggy_forget_one(n):
    r = math.isqrt(n)
    if r * r != n:
        return False
    d = 2
    while d * d <= r:            # với r = 1: 2*2 <= 1 sai -> không vào vòng
        if r % d == 0:
            return False
        d += 1
    return True                  # trả True cho n = 1!

print("buggy n=1:", buggy_forget_one(1), "| đúng:", is_three(1))

buggy n=1: True | đúng: False


### Lỗi 3 — Dùng `int(n ** 0.5)` với số lớn

Với `n` nhỏ như đề thì không sao, nhưng với số rất lớn, phép căn số thực có thể sai một đơn vị.

In [20]:
big = (10 ** 8 + 7) ** 2 - 1                 # ngay dưới một số chính phương lớn
print(int(big ** 0.5), math.isqrt(big))      # float làm tròn LÊN thành 100000007 (sai), isqrt cho 100000006 (đúng)

100000007 100000006


### Lỗi 4 — Đếm cặp ước nhưng cộng 2 cả khi `d == n // d`

Với số chính phương, ước `√n` bị đếm hai lần → `4` có "4 ước" thay vì 3 → trả `False` sai.

In [21]:
def buggy_double_count(n):
    count, d = 0, 1
    while d * d <= n:
        if n % d == 0:
            count += 2            # SAI khi d == n // d
        d += 1
    return count == 3

print("buggy n=4:", buggy_double_count(4), "| đúng:", is_three(4))

buggy n=4: False | đúng: True


Bản lỗi **không bao giờ** trả `True` (số đếm luôn chẵn).

### Lỗi 5 — Vòng `while d * d < n` (dùng `<` thay `<=`)

Bỏ sót `d = √n`, đúng ước quan trọng nhất của bài này.

## 14. Mẹo nhận dạng pattern

- "Đếm/kiểm tra **ước số**" → duyệt tới `√n` theo cặp `(d, n//d)`, chú ý cặp trùng.
- "Số ước **lẻ**" ⇔ **số chính phương** — một sự thật rất hay gặp (LeetCode 319 Bulb Switcher).
- "Đúng `k` ước" → nghĩ tới dạng phân tích thừa số: số ước của `p₁^a₁ · p₂^a₂ ⋯` là `(a₁+1)(a₂+1)⋯`. `k = 3` (nguyên tố) ⇒ chỉ một thừa số với `a = 2`.
- Ràng buộc nhỏ + nhiều truy vấn → **tiền xử lý** (sàng) rồi tra `set`.

## 15. Biến thể

### 15.1. Đếm số `n ≤ N` có đúng 3 ước

Bằng số số nguyên tố `p` với `p² ≤ N`, tức số nguyên tố ≤ `isqrt(N)`.

In [22]:
def count_three_divisors_upto(N):
    return len(sieve(math.isqrt(N)))

print(count_three_divisors_upto(30), count_three_divisors_upto(10 ** 4), count_three_divisors_upto(10 ** 12))

3 25 78498


### 15.2. LeetCode 1390 — Four Divisors (ý tưởng)

Với một số có **đúng 4 ước**, trả tổng các ước. Cách làm: đếm cặp ước tới `√n`, dừng khi vượt 4.

In [23]:
def sum_if_four_divisors(n):
    divs = []
    d = 1
    while d * d <= n:
        if n % d == 0:
            divs.append(d)
            if d != n // d:
                divs.append(n // d)
            if len(divs) > 4:
                return 0
        d += 1
    return sum(divs) if len(divs) == 4 else 0

print(sum_if_four_divisors(21), sum_if_four_divisors(4), sum_if_four_divisors(8))   # 32 0 15

32 0 15


`21 = 3·7` → ước `1, 3, 7, 21` (tổng 32). `8 = 2³` → `1, 2, 4, 8` (tổng 15). Số có đúng 4 ước là `p·q` (hai nguyên tố khác nhau) hoặc `p³`.

### 15.3. Số ước bằng công thức phân tích thừa số

In [24]:
def count_divisors_factor(n):
    total, p = 1, 2
    while p * p <= n:
        a = 0
        while n % p == 0:
            n //= p
            a += 1
        total *= a + 1
        p += 1
    if n > 1:              # phần còn lại là một số nguyên tố
        total *= 2
    return total

for n in [1, 4, 12, 36, 49, 97]:
    print(n, count_divisors_factor(n), count_divisors_sqrt(n))

1 1 1
4 3 3
12 6 6
36 9 9
49 3 3
97 2 2


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `math.isqrt(24)`, `math.isqrt(25)`, `list(range(4, 20, 4))` là gì?

**Bài 2 (tính tay):** liệt kê ước của `25`, `27`, `121`. Số nào có đúng 3 ước?

**Bài 3 (điền chỗ trống):** hoàn thiện đếm cặp ước.

In [25]:
def exercise_count_divisors(n):
    count, d = 0, 1
    while d * d <= n:
        if n % d == 0:
            # TODO: cộng 1 hay 2 tuỳ d có bằng n // d không
            pass
        d += 1
    return count

# assert exercise_count_divisors(16) == 5
# assert exercise_count_divisors(49) == 3

**Bài 4 (tự viết lại):** viết `exercise_is_three(n)` theo insight `p²`, nhớ chặn `n = 1`.

In [26]:
def exercise_is_three(n):
    # TODO
    pass

# assert exercise_is_three(4) is True
# assert exercise_is_three(1) is False
# assert exercise_is_three(16) is False

**Bài 5 (biến thể):** viết `exercise_has_odd_divisors(n)` trả `True` khi `n` có **số lẻ** ước (không đếm ước).

**Bài 6 (mini challenge):** viết `exercise_three_divisor_numbers(lo, hi)` trả list mọi số trong `[lo, hi]` có đúng 3 ước, tăng dần, dùng sàng.

In [27]:
def exercise_has_odd_divisors(n):
    # TODO
    pass

def exercise_three_divisor_numbers(lo, hi):
    # TODO
    pass

# assert exercise_has_odd_divisors(36) is True and exercise_has_odd_divisors(12) is False
# assert exercise_three_divisor_numbers(1, 50) == [4, 9, 25, 49]
# assert exercise_three_divisor_numbers(100, 200) == [121, 169]

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | isqrt làm tròn xuống | 4² = 16 ≤ 24 < 25 | `range` dừng **trước** 20 |
| 2 | cặp ước | 27 = 3³ | 25 và 121 có 3 ước |
| 3 | cặp trùng | `d * d == n` | `count += 1 if d * d == n else 2` |
| 4 | hai điều kiện | chính phương + căn nguyên tố | `r = isqrt(n)`; `r*r == n and is_prime(r)` |
| 5 | số ước lẻ ⇔ ? | chính phương | `math.isqrt(n) ** 2 == n` |
| 6 | p² trong `[lo, hi]` | sàng tới `isqrt(hi)` | `[p*p for p in sieve(isqrt(hi)) if p*p >= lo]` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `4`, `5`, `[4, 8, 12, 16]`.

**Bài 2:** `25`: 1, 5, 25 (3 ước ✔). `27`: 1, 3, 9, 27 (4 ước). `121`: 1, 11, 121 (3 ước ✔).

In [28]:
print(math.isqrt(24), math.isqrt(25), list(range(4, 20, 4)))
print([d for d in range(1, 28) if 27 % d == 0], is_three(25), is_three(27), is_three(121))


def answer_count_divisors(n):
    count, d = 0, 1
    while d * d <= n:
        if n % d == 0:
            count += 1 if d * d == n else 2
        d += 1
    return count


def answer_is_three(n):
    r = math.isqrt(n)
    return r * r == n and is_prime(r)


def answer_has_odd_divisors(n):
    return math.isqrt(n) ** 2 == n


def answer_three_divisor_numbers(lo, hi):
    return [p * p for p in sieve(math.isqrt(hi)) if p * p >= lo]


assert answer_count_divisors(16) == 5
assert answer_count_divisors(49) == 3
assert answer_is_three(4) is True
assert answer_is_three(1) is False
assert answer_is_three(16) is False
assert answer_has_odd_divisors(36) is True and answer_has_odd_divisors(12) is False
assert answer_three_divisor_numbers(1, 50) == [4, 9, 25, 49]
assert answer_three_divisor_numbers(100, 200) == [121, 169]
for n in range(1, 3000):
    assert answer_is_three(n) == is_three_naive(n)
    assert answer_has_odd_divisors(n) == (count_divisors_sqrt(n) % 2 == 1)
print("Đáp án tham khảo PASS toàn bộ test!")

4 5 [4, 8, 12, 16]
[1, 3, 9, 27] True False True


Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
import math

class Solution:
    def isThree(self, n: int) -> bool:
        r = math.isqrt(n)
        if r * r != n or r < 2:          # phải là p², và loại n = 1
            return False
        return all(r % d for d in range(2, math.isqrt(r) + 1))   # r nguyên tố
```

**Câu thần chú:** *đúng 3 ước ⇔ bình phương của một số nguyên tố.*

**Checklist trước khi submit:**
- [ ] `n = 1` trả `False`?
- [ ] Kiểm tra cả "chính phương" **và** "căn là nguyên tố" (16 → False)?
- [ ] Dùng `math.isqrt`, không dùng `int(n ** 0.5)` cho số lớn?
- [ ] Nếu đếm cặp ước: cặp trùng chỉ cộng 1, điều kiện vòng là `d * d <= n`?